# Batch: reviewed natural questions v2

Run the 20 reviewed original r/NISTControls questions separately from the 136-row benchmark and the 30-row retest. Select **Runtime → Change runtime type → A100 GPU**, then **Run all**. Setup may request Drive access.

The author-reviewed questions, reference labels, protocol and executable fingerprint were published before inference at `71a9a187d0a3fe0d64d71cdefbacdb345cce890d`. Three original-post captures were completed in this version. Keep all 20 questions: three fully answerable from the catalog, 12 partially answerable and five outside its scope. Only two require actual ODP values. These are small exploratory denominators, and the reference labels are not independent expert validation. The launcher creates an isolated Python 3.12.14 environment compatible with the frozen dependencies, independently of the Colab notebook kernel version.

Both answer paths share the pinned Qwen2.5-7B-Instruct BF16 model and matched evidence. Outputs, all generation attempts, runtime identity and logs are retained in Drive. Automatic completion is followed by a separate author review of 40 question/path pairs.


In [6]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
from google.colab import drive

drive.mount('/content/drive')
SOURCE_COMMIT = '71a9a187d0a3fe0d64d71cdefbacdb345cce890d'
REPO_DIR = Path('/content/ComplianceGPT_natural_v2')
STUDY_RELATIVE = Path('experiments/external_validity/natural_questions_v2')
OUTPUT_DIR = Path('/content/drive/MyDrive/ComplianceGPT/experiments/natural_questions_v2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CONSOLE_LOG = OUTPUT_DIR / 'console.log'
LAUNCHER_LOG = OUTPUT_DIR / 'launcher_steps.jsonl'
test_environment = os.environ.copy()
test_environment.update(USE_TORCH='1', USE_TF='0', USE_FLAX='0',
                        TOKENIZERS_PARALLELISM='false', HF_HUB_DISABLE_TELEMETRY='1',
                        CUBLAS_WORKSPACE_CONFIG=':4096:8')
test_environment['PYTHONPATH'] = str(REPO_DIR / 'src') + ':' + str(REPO_DIR)

def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def run_logged(command, cwd=None):
    event = {'command': list(command), 'cwd': str(cwd) if cwd else None,
             'started_at_utc': datetime.now(timezone.utc).isoformat()}
    return_code = None
    try:
        with CONSOLE_LOG.open('a', encoding='utf-8') as log:
            log.write('\nAttempt: ' + event['started_at_utc'] + '\n' + repr(list(command)) + '\n')
            log.flush()
            with subprocess.Popen(command, cwd=cwd, env=test_environment,
                                  stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                  text=True, bufsize=1) as process:
                for line in process.stdout:
                    print(line, end='')
                    log.write(line)
                    log.flush()
                return_code = process.wait()
        if return_code:
            raise RuntimeError('Command failed. The Drive console log and existing checkpoints are retained. Review the failure before resuming.')
    finally:
        event.update(return_code=return_code, finished_at_utc=datetime.now(timezone.utc).isoformat())
        with LAUNCHER_LOG.open('a', encoding='utf-8') as stream:
            stream.write(json.dumps(event, ensure_ascii=False) + '\n')
            stream.flush()
            os.fsync(stream.fileno())

print('Results and checkpoints:', OUTPUT_DIR)
print('Python:', platform.python_version())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Results and checkpoints: /content/drive/MyDrive/ComplianceGPT/experiments/natural_questions_v2
Python: 3.13.15


In [7]:
if not REPO_DIR.exists():
    run_logged(['git', 'clone', '--filter=blob:none', '--no-checkout',
                'https://github.com/Wenjia1215/ComplianceGPT.git', str(REPO_DIR)])
    run_logged(['git', 'fetch', '--depth=1', 'origin', SOURCE_COMMIT], cwd=REPO_DIR)
    run_logged(['git', 'checkout', '--detach', SOURCE_COMMIT], cwd=REPO_DIR)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
if head != SOURCE_COMMIT:
    raise RuntimeError('This working directory has a different commit. Use a fresh Colab runtime; preserve the existing Drive result folder.')
if subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=no'], cwd=REPO_DIR, text=True):
    raise RuntimeError('The registered checkout contains modified source files.')
run_logged(['git', 'show', '-s', '--format=fuller', 'HEAD'], cwd=REPO_DIR)
run_logged([sys.executable, '-u', str(STUDY_RELATIVE / 'run_natural_questions.py'),
            '--output-dir', str(OUTPUT_DIR), '--preflight-only'], cwd=REPO_DIR)
print('Exact public question/label/code registration passed, before model loading.')


commit 71a9a187d0a3fe0d64d71cdefbacdb345cce890d
Author:     WenjiaWang <65235107+Wenjia1215@users.noreply.github.com>
AuthorDate: Mon Oct 5 11:20:01 2026 -0400
Commit:     WenjiaWang <65235107+Wenjia1215@users.noreply.github.com>
CommitDate: Mon Oct 5 11:20:01 2026 -0400

    Freeze reviewed natural-question v2 inputs and author reference labels
{"registered_questions": 20, "source_commit": "71a9a187d0a3fe0d64d71cdefbacdb345cce890d", "public_registration": "https://raw.githubusercontent.com/Wenjia1215/ComplianceGPT/71a9a187d0a3fe0d64d71cdefbacdb345cce890d/experiments/external_validity/natural_questions_v2/registration.json", "inference_performed": false}
Exact public question/label/code registration passed, before model loading.


In [8]:
run_logged(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'])
# NumPy 2.0.2 supports Python through 3.12. Keep its frozen version instead
# of changing the scientific stack to accommodate a newer Colab kernel.
# https://github.com/numpy/numpy/releases/tag/v2.0.2
run_logged([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.12.19'])
UV_COMMAND = [sys.executable, '-m', 'uv']
RUNTIME_DIR = Path('/content/ComplianceGPT_natural_v2_py312')
EXPERIMENT_PYTHON = str(RUNTIME_DIR / 'bin/python')
if not Path(EXPERIMENT_PYTHON).exists():
    run_logged(UV_COMMAND + ['venv', '--python', '3.12.14', '--seed', str(RUNTIME_DIR)])
actual_python = subprocess.check_output([EXPERIMENT_PYTHON, '-c',
                                        'import platform; print(platform.python_version())'],
                                       env=test_environment, text=True).strip()
if actual_python != '3.12.14':
    raise RuntimeError('The study interpreter must be Python 3.12.14. Use a fresh Colab runtime.')
# The new environment has no Colab media or TensorFlow integrations.
run_logged([EXPERIMENT_PYTHON, '-m', 'pip', 'install', '-q', 'torch==2.6.0',
            '--index-url', 'https://download.pytorch.org/whl/cu124'])
run_logged([EXPERIMENT_PYTHON, '-m', 'pip', 'install', '-q', '-r',
            str(REPO_DIR / STUDY_RELATIVE / 'requirements-colab.txt')])
run_logged([EXPERIMENT_PYTHON, '-m', 'pip', 'freeze'])
print('Pinned text-model stack installed in the isolated Python', actual_python, 'environment.')


NVIDIA A100-SXM4-40GB, 40960 MiB, 580.82.07
accelerate==1.12.0
certifi==2026.7.22
charset-normalizer==3.5.2
cloudpickle==3.1.2
faiss-cpu==1.10.0
filelock==3.32.3
fsspec==2026.7.0
huggingface-hub==0.30.2
idna==3.20
Jinja2==3.1.6
joblib==1.6.0
MarkupSafe==3.0.3
mpmath==1.3.0
narwhals==2.26.0
networkx==3.6.1
numpy==2.0.2
nvidia-cublas-cu12==12.4.5.8
nvidia-cuda-cupti-cu12==12.4.127
nvidia-cuda-nvrtc-cu12==12.4.127
nvidia-cuda-runtime-cu12==12.4.127
nvidia-cudnn-cu12==9.1.0.70
nvidia-cufft-cu12==11.2.1.3
nvidia-curand-cu12==10.3.5.147
nvidia-cusolver-cu12==11.6.1.9
nvidia-cusparse-cu12==12.3.1.170
nvidia-cusparselt-cu12==0.6.2
nvidia-nccl-cu12==2.21.5
nvidia-nvjitlink-cu12==12.4.127
nvidia-nvtx-cu12==12.4.127
packaging==26.3
pandas==2.2.3
pillow==12.3.0
psutil==7.2.2
python-dateutil==2.9.0.post0
pytz==2026.5
PyYAML==6.0.3
regex==2026.9.29
requests==2.34.2
safetensors==0.5.3
scikit-learn==1.9.1
scipy==1.18.1
sentence-transformers==4.1.0
setuptools==78.1.0
six==1.17.0
sympy==1.13.1
threadpoo

In [9]:
model_preflight = '''
import importlib.util
import torch
unexpected = [name for name in ('timm', 'torchvision', 'torchaudio') if importlib.util.find_spec(name)]
if unexpected:
    raise RuntimeError('Optional media backends remain: ' + ', '.join(unexpected))
if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
    raise RuntimeError('Select a CUDA GPU supporting BF16, preferably A100.')
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoModel, AutoModelForSequenceClassification, BertConfig, XLMRobertaConfig
common = dict(vocab_size=32, hidden_size=8, num_hidden_layers=1,
              num_attention_heads=2, intermediate_size=16)
AutoModel.from_config(BertConfig(**common))
AutoModelForSequenceClassification.from_config(XLMRobertaConfig(**common))
print('Text-model imports and tiny loading paths passed.')
print('Actual GPU:', torch.cuda.get_device_name(0), 'Torch:', torch.__version__, 'CUDA:', torch.version.cuda)
'''
run_logged([EXPERIMENT_PYTHON, '-c', model_preflight], cwd=REPO_DIR)
run_logged([EXPERIMENT_PYTHON, '-m', 'unittest', 'tests.test_natural_questions',
            'tests.test_natural_questions_v2', 'tests.test_evidence_window', '-q'], cwd=REPO_DIR)
print('Offline review, scoring and shared-window checks passed. No study inference yet.')


Text-model imports and tiny loading paths passed.
Actual GPU: NVIDIA A100-SXM4-40GB Torch: 2.6.0+cu124 CUDA: 12.4
----------------------------------------------------------------------
Ran 39 tests in 0.627s

OK
[QUR] RUN_MODE=lib
[QUR] Library mode loaded. (Batch generation & Tests disabled)
QUR library exported as QUR_LIB
[QUR] RUN_MODE=lib
Offline review, scoring and shared-window checks passed. No study inference yet.


The next cell performs the registered experiment. Keep the original questions and frozen labels unchanged. A failed attempt retains logs and per-question checkpoints. Resume only with this same source commit and compatible recorded runtime. Do not delete failed calls or rerun to select favorable outputs.


In [10]:
import json
from urllib.request import urlopen

url = (
    "https://raw.githubusercontent.com/Wenjia1215/ComplianceGPT/"
    "73df3a31720e3ce223bd64839802ffaa8079b8d6/"
    "experiments/external_validity/natural_questions_v2/"
    "Batch_Natural_Questions_v2.ipynb"
)
with urlopen(url, timeout=30) as response:
    fixed_notebook = json.load(response)

exec("".join(fixed_notebook["cells"][6]["source"]), globals())

From https://github.com/Wenjia1215/ComplianceGPT
 * branch            27c437a2176847b6ac82f86b3f1abbb3ba63c1ac -> FETCH_HEAD
Previous HEAD position was 71a9a18 Freeze reviewed natural-question v2 inputs and author reference labels
HEAD is now at 27c437a Register natural-question authority validation repair before answer generation
{"registered_questions": 20, "source_commit": "27c437a2176847b6ac82f86b3f1abbb3ba63c1ac", "public_registration": "https://raw.githubusercontent.com/Wenjia1215/ComplianceGPT/27c437a2176847b6ac82f86b3f1abbb3ba63c1ac/experiments/external_validity/natural_questions_v2/registration.json", "inference_performed": false}
----------------------------------------------------------------------
Ran 53 tests in 1.057s

OK
[QUR] RUN_MODE=lib
[QUR] Library mode loaded. (Batch generation & Tests disabled)
QUR library exported as QUR_LIB
[QUR] RUN_MODE=lib
Prior retrieval attempt retained: /content/drive/MyDrive/ComplianceGPT/experiments/natural_questions_v2
Repaired register

In [11]:
import csv
summary = json.loads((OUTPUT_DIR / 'summary.json').read_text())
config = json.loads((OUTPUT_DIR / 'run_config.json').read_text())
outcomes = [json.loads(s) for s in (OUTPUT_DIR / 'per_question_outcomes.jsonl').read_text().splitlines() if s.strip()]
assert summary['question_count'] == 20 and summary['paired_generation_count'] == 40
assert len(outcomes) == 40 and len({r['query_id'] for r in outcomes}) == 20
assert summary['scope_counts'] == {'full': 3, 'partial': 12, 'outside': 5}
assert summary['odp_label_counts'] == {'required': 2, 'not_required': 13, 'not_applicable': 5}
assert config['signature']['source_commit'] == SOURCE_COMMIT
assert summary['experiment_complete'] is False
assert summary['independent_correctness_validated'] is False
with (OUTPUT_DIR / 'post_run_semantic_review.csv').open(newline='', encoding='utf-8') as stream:
    assert len(list(csv.DictReader(stream))) == 40
for system in ('compliancegpt', 'generative_baseline'):
    metrics = summary['systems'][system]
    assert metrics['strict_full_catalog_contract']['denominator'] == 3
    assert metrics['odp_author_reference_sensitivity']['denominator'] == 2
    print(system, json.dumps(metrics, indent=2))
print('All 20 paired questions retained. Strict denominator: 3; positive ODP denominator: 2.')
print('The separate scope/responsiveness review must inspect all 40 answers, including actual clarification wording.')

snapshot = OUTPUT_DIR / 'source_snapshot'
registration = json.loads((REPO_DIR / STUDY_RELATIVE / 'registration.json').read_text())
source_paths = {str(STUDY_RELATIVE / name) for name in registration['file_sha256']}
source_paths.update(registration['code_sha256'])
source_paths.update(str(STUDY_RELATIVE / name) for name in ('registration.json', 'PROTOCOL.md', 'README.md', 'QUESTIONS.md'))
for relative in sorted(source_paths):
    target = snapshot / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(REPO_DIR / relative, target)
(OUTPUT_DIR / 'launcher_identity.json').write_text(json.dumps({
    'source_commit': SOURCE_COMMIT, 'kernel_python': platform.python_version(),
    'experiment_python': config['signature']['python'], 'experiment_interpreter': EXPERIMENT_PYTHON,
    'registration_sha256': digest(REPO_DIR / STUDY_RELATIVE / 'registration.json'),
    'automatic_pair_count': 40, 'post_run_author_review': 'pending',
    'final_notebook': 'Save the fully executed notebook from the Colab File menu after Run all finishes.'
}, indent=2) + '\n')


compliancegpt {
  "clause_id_group_coverage": {
    "denominator": 15,
    "numerator": 6,
    "rate": 0.4,
    "wilson_95": [
      0.1982449614134158,
      0.6425316987937539
    ]
  },
  "complete_clause_text_group_coverage": {
    "denominator": 15,
    "numerator": 6,
    "rate": 0.4,
    "wilson_95": [
      0.1982449614134158,
      0.6425316987937539
    ]
  },
  "odp_author_reference_sensitivity": {
    "denominator": 2,
    "numerator": 2,
    "rate": 1.0,
    "wilson_95": [
      0.34238022750665303,
      1.0
    ]
  },
  "odp_author_reference_specificity": {
    "denominator": 13,
    "numerator": 1,
    "rate": 0.07692307692307693,
    "wilson_95": [
      0.013710421242556586,
      0.3331395092109959
    ]
  },
  "runtime_contract_valid": {
    "denominator": 20,
    "numerator": 20,
    "rate": 1.0,
    "wilson_95": [
      0.8388748419471806,
      1.0
    ]
  },
  "strict_full_catalog_contract": {
    "denominator": 3,
    "numerator": 0,
    "rate": 0.0,
    "wilso

479

In [12]:
hashes = {str(p.relative_to(OUTPUT_DIR)): digest(p)
          for p in sorted(OUTPUT_DIR.rglob('*')) if p.is_file() and p.name != 'SHA256SUMS.json'}
(OUTPUT_DIR / 'SHA256SUMS.json').write_text(json.dumps(hashes, indent=2, sort_keys=True) + '\n')
archive_path = Path(shutil.make_archive(str(OUTPUT_DIR.parent / 'natural_questions_v2_results'),
                                      'zip', root_dir=OUTPUT_DIR))
print('Result package:', archive_path)
print('Package SHA-256:', digest(archive_path))
print('Return this ZIP plus the fully executed notebook (File → Download → Download .ipynb).')
print('Keep all cell outputs. The returned notebook will be archived with the results in GitHub.')
from google.colab import files
files.download(str(archive_path))


Result package: /content/drive/MyDrive/ComplianceGPT/experiments/natural_questions_v2_results.zip
Package SHA-256: 0636a22abf3ea4a63d432e04f53bd61e884d067ad41de73beaf25532947b63fb
Return this ZIP plus the fully executed notebook (File → Download → Download .ipynb).
Keep all cell outputs. The returned notebook will be archived with the results in GitHub.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

After execution, download this notebook with all cell outputs preserved and return it with `natural_questions_v2_results.zip`. The next review checks source/model/context identities, paired completeness, actual responses and unsupported claims before dissertation integration. The experiment is not complete merely because automatic scoring succeeded.
